## **Historical Perspectives on Current Economic Issues: Big Data and Applications**
### *AI/ML for language and vision*

Torben Skov Dyg Johansen (tsdj@sam.sdu.dk, tsdj@rooftop-analytics.com)

Assistant Professor of Econometrics and Data Science at SDU and Data Scientist at Rooftop Analytics

https://torbensdjohansen.github.io/

### Outline

#### Language
1. Tokenization: Representing text as numbers
1. Text embeddings
1. Neural networks for text data
1. Text classification from scratch

#### Vision
1. Document classification
1. Segmentations
1. Handwritten text recognition
1. **TODO application**

# Tokenization

**Computers only like numbers**: We need to somehow represent text as numbers

*We essentially do this the same way we handle categorical variables in regressions*

## Tokenizing a sentence

Tokenization works by building a dictionary for each word (or more generally token, it could be the sub-word or even character level).

For example, we may build the following dictionary:

In [54]:
tokenization_dict = {
    'the': 0,
    'cat': 1,
    'sat': 2,
    'on': 3,
    'mat': 4,
}

example_text = 'the cat sat on the mat'
tokenized_text = [tokenization_dict[word] for word in example_text.split()]
print(tokenized_text)

[0, 1, 2, 3, 0, 4]


## One-hot encoding

It makes little sense to think of separate words as lying on a continuous scale -- and thus the numbers on the previous slide should be thought of as representing a categorical variable

As with using dummies in regressions, we now expand this categorial variable to a number of different 0/1 variables, which we call one-hot encoding

In [59]:
import torch

one_hot_encoded = torch.nn.functional.one_hot(torch.tensor(tokenized_text))
print(one_hot_encoded)

tensor([[1, 0, 0, 0, 0],
        [0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 1, 0],
        [1, 0, 0, 0, 0],
        [0, 0, 0, 0, 1]])


This matrix has five columns, as we have a dictionary with five unique words, and six rows, as we have a sentence consisting of six words (of which two are the same)

If we then estimated a linear model using OLS, we could make our preditions like so

In [65]:
betas = torch.tensor([1, 2, 3, 5, 9])

one_hot_encoded @ betas

tensor([1, 2, 3, 5, 1, 9])

Typically, a neural network will consists of a series of such matrix multiplications followed by some non-linearity

**Note**: It's quite inefficient to perform the above matrix multiplication, as the first matrix is *sparse* -- and in practice, we'd implement this using a lookup

In [69]:
betas_lookup = dict(zip(tokenization_dict.values(), betas))
torch.tensor([betas_lookup[token] for token in tokenized_text])

tensor([1, 2, 3, 5, 1, 9])

**Note**: Often, we'll associate a *vector* (rather than just one number) with each token

# Text embeddings

**TODO potentially place this after *Neural networks for text***

Our initial task of transforming our tokenized text into numbers is generally performed by an embedding layer -- which is just a lookup table, but where the numbers to look up are optimized as part of the neural network

# Neural networks for text

We now know how to represent text as numerical data suitable for modelling. Our next objective is to learn *how* to model text data

One key observation is that text is somewhat similar to time series data, in that a **TODO sequential, correlation to nearby, ...**

Modern approaches roughly falls into one of two categories
1. Recurrent neural networks (RNNs; "classical" approach)
1. Transformers (SOTA)

We'll first consider RNNs before turning to transformer models

### A current cell of a neural network

<img src="./figs/figure_6-9.png" alt="Drawing" style="width: 400px;"/>
Source: Deep Learning with Python by Francois Collet (ISBN10: 9781617294433)

We have not yet specified the type of recurrent connection, just that it has form

$y_t = f(x_t, y_{t - 1})$

We can think about an autoregressive linear model with covariates as one example

$y_t = \alpha y_{t - 1} + \beta f_t$

### Recurrent cells and unfolding

Usually, a recurrent neural network looks something like the image below - i.e., it has a "loop" where it feeds into itself (left). This can then be "unfolded" (right), in which way it looks more like a regular network, with an additional input and output ("hidden states").

<img src="./figs/Recurrent_neural_network_unfold.svg" alt="Drawing" style="width: 600px;"/>
Source: "fdeloche - Own work, CC BY-SA 4.0, https://commons.wikimedia.org/w/index.php?curid=60109157"

# Text classification: HISCO classifier

 Let's build a simple HISCO classifier using the tools we **TODO continue** 

### Building a character-level tokenizer

In [23]:
from functools import partial

import torch

from torch import Tensor
from torch.utils.data import Dataset

import pandas as pd

CHARS_IN_TOYDATA = [' ', '"', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', '@', '[', ']', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '¢', '£', '©', '¬', 'Â', 'Ã', 'â', 'œ', 'š', 'ž', '‚', '„', '€']
MAP_CHAR_IDX = {char: idx for idx, char in enumerate(CHARS_IN_TOYDATA, start=2)}

def tokenize(hisco: str, max_len: int) -> list[int]:
    encoded = [MAP_CHAR_IDX.get(char, 0) for char in hisco]
    encoded = encoded[:max_len]
    encoded += [1] * (max_len - len(encoded))

    return encoded

*Illustrating tokenizing occupational descriptions*

In [26]:
for char, token in zip('farmer', tokenize('farmer', 6)):
    print(f'Mapping "{char}" to "{token}"')

Mapping "f" to "36"
Mapping "a" to "31"
Mapping "r" to "48"
Mapping "m" to "43"
Mapping "e" to "35"
Mapping "r" to "48"


[39, 48, 45, 44, 2, 53, 45, 48, 41, 49, 2, 42, 31, 32, 45, 51, 48, 35, 48, 1]

In [27]:
tokenize('iron works labourer', 20) # longer example

[39, 48, 45, 44, 2, 53, 45, 48, 41, 49, 2, 42, 31, 32, 45, 51, 48, 35, 48, 1]

### Building a HISCO dataset

In [22]:
class HISCODataset(Dataset):
    def __init__(self, dataset: pd.DataFrame):
        super().__init__()

        self.dataset = dataset
        self.tokenizer = partial(tokenize, max_len=32)

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, item: int) -> dict[str, str | Tensor]:
        record = self.dataset.iloc[item]
        encoded = self.tokenizer(record.occ1)

        package = {
            'occ1': record.occ1,
            'encoded': torch.tensor(encoded, dtype=torch.long),
            'label': torch.tensor(record.label, dtype=torch.long),
        }

        return package

In [30]:
import os
from dirs import DATA_DIR

train_data = pd.read_csv(os.path.join(DATA_DIR, 'toy_data_train.csv'))
train_data.head(3)

,occ1,label
0,iron works labourer,1915
1,fireman midland railway,1885
2,ironstone mine foreman,513


In [31]:
train_dataset = HISCODataset(train_data)
train_dataset[0]

{'occ1': 'iron works labourer',
 'encoded': tensor([39, 48, 45, 44,  2, 53, 45, 48, 41, 49,  2, 42, 31, 32, 45, 51, 48, 35,
         48,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1]),
 'label': tensor(1915)}

*So which HISCO code does the label 1915 refer to?*

In [35]:
from histocc import DATASETS

keys = DATASETS['keys']()
map_hisco_label = dict(keys[['hisco', 'code']].values)
map_label_hisco = {v: k for k, v in map_hisco_label.items()}

map_label_hisco[1915]

99910

*... and some other examples..*

In [39]:
for description, label in train_data.head(5).values:
    hisco = map_label_hisco[label]
    print(f'"{description}" has label {label} -> HISCO: {hisco}')

"iron works labourer" has label 1915 -> HISCO: 99910
"fireman midland railway" has label 1885 -> HISCO: 98330
"ironstone mine foreman" has label 513 -> HISCO: 22610
"engineer fitter and turner" has label 1321 -> HISCO: 83320
"carpenter deceased" has label 1751 -> HISCO: 95410


With our dataset class, we can build data loaders, which are wrappers around our dataset responsible for creating a **batch** of data

In [45]:
from torch.utils.data import DataLoader

test_data = pd.read_csv(os.path.join(DATA_DIR, 'toy_data_test.csv'))
test_dataset = HISCODataset(test_data)

train_data_loader = DataLoader(train_dataset, batch_size=32)
test_data_loader = DataLoader(test_dataset, batch_size=32)

### Building a HISCO classifier

**Goal**: Given an occupational description, provide its associated HISCO

In [40]:
from torch import Tensor, nn


class HISCOClassifier(nn.Module):
    def __init__(self, vocab_size: int = 100, hidden_size: int = 128):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.classifier = nn.Linear(hidden_size, 1919)

    def forward(self, input_seq: Tensor) -> Tensor:
        out = self.embedding(input_seq)
        out, _ = self.gru(out)
        out = out[:, -1, :]
        out = self.classifier(out)

        return out


We can can instantiate our **model**; we also need an objective (or **loss**) function and an **optimizer**

In [42]:
model = HISCOClassifier()

# For single-label classification, cross entropy is the
# standard loss function to use. This has the limitation
# that it only allows ONE HISCO code for an occupational
# description
loss_fn = torch.nn.CrossEntropyLoss()

# Adam/AdamW are common types of optimizers. An optimizer
# is responsible for how we choose, based on derivatives
# with respect to our loss function, to update the para-
# meters of our model
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

### Building a training loop

We now train our model by iteratively
1. Draw a **batch** of descriptions and labels
1. Make our model *predict* the labels given the descriptions
1. Calculate the loss wrt. the predictions and labels
1. Calculate the derivative of the loss wrt. the model parameters
1. Update the model parameters

In [43]:
def train_epoch(model, optimizer, data_loader, loss_fn):
    model.train()

    for batch in data_loader:
        optimizer.zero_grad()
        
        out = model(batch['encoded']) # make predictions
        loss = loss_fn(out, batch['label']) # calculate loss
        loss.backward() # calculate derivatives

        optimizer.step() # update network parameters

### Building an evaluation loop

Ultimately, we want to evaluate our model in terms of how often it corretly predicts a HISCO code given an occupational description

In [44]:
@torch.no_grad
def evaluate(model, data_loader):
    model.eval()

    total_correct = 0 # keep count of correct predictions
    total_count = 0 # keep count of total number of predictions

    for batch in data_loader:
        out = model(batch['encoded']).argmax(1)

        total_correct += (out == batch['label']).sum().item()
        total_count += batch['label'].size(0)

    return total_correct / total_count # calculate accuracy

### Training our HISCO classifier

We now train our model by looping over the training data 10 times

In [49]:
for epoch in range(1, 11):
    train_epoch(model, optimizer, train_data_loader, loss_fn)
    acc = evaluate(model, test_data_loader)
    
    print(f'Trained for {epoch} epochs. Validation accuracy: {100 * acc}%')

Trained for 1 epochs. Validation accuracy: 73.8%
Trained for 2 epochs. Validation accuracy: 73.4%
Trained for 3 epochs. Validation accuracy: 74.6%
Trained for 4 epochs. Validation accuracy: 75.2%
Trained for 5 epochs. Validation accuracy: 74.1%
Trained for 6 epochs. Validation accuracy: 75.7%
Trained for 7 epochs. Validation accuracy: 75.9%
Trained for 8 epochs. Validation accuracy: 77.2%
Trained for 9 epochs. Validation accuracy: 77.5%
Trained for 10 epochs. Validation accuracy: 76.4%


*if we were to always predict the most common class, we would only achieve 10% accuracy*

**TODO EXERCISE HERE, EXPERIMENTING ON TOY DATASET**

# Transcription of Handwritten Data

**TODO potentially motivation ehre*

## Document Digitization

**Ambition**: Make any source of handwritten data as readily available for analysis as register data

<img src="./figs/pipeline.jpg" alt="Drawing" style="width: 800px;"/>

## From Source to Output

<img src="./figs/hana-transcription.png" alt="Drawing" style="width: 800px;"/>

#### Visition Transformer Encoder/Transformer Decoder

<img src="./figs/vit_enc_transformer_dec_v2.png" alt="Drawing" style="width: 800px;"/>

## Prerequisites

## Performance: Dates

## Performance: Names

## Copenhagen Infant Health Visitor Records

## Link-Lives

**TODO maybe incl**